In [10]:
import pandas as pd



saldo_historico=pd.read_excel("Rachas.xlsx", sheet_name='historia')
retiros=pd.read_excel("Rachas.xlsx", sheet_name='retiros')

saldo_historico

,identificacion,corte_mes,saldo
0,R6XC1HPW2F2OYGFS2,2024-12-31,3983478
1,7AQPGU50TED28A7G6,2024-12-31,4163361
2,EX2GMK7UHXBM6P2OW,2023-07-31,6760569
3,V1SRV6QGJWWC8JVW0,2023-02-28,1461567
4,PQHMTD7TS2Q7AM9C6,2023-09-30,4992649
...,...,...,...
2920,KLRH1ZESF492S056H,2024-11-30,2552608
2921,VIOMZ3660T8XTY6RC,2023-04-30,6991126
2922,H9QCKG6XB01NQCZO9,2024-04-30,3277504
2923,LATSV8PKLN0G0XSCQ,2023-02-28,4357563


In [11]:
retiros

,identificacion,fecha_retiro
0,0TTW5R9RRCJ0A9E5F,2024-10-10
1,1LFTDYT2H0I9ZGED3S8,2023-11-11
2,6T645BU8TYM8VGXKO,2025-01-10
3,869YG73INQB5W1NQT,2024-12-05
4,THJ9OBJH3W6ANRCMS,2024-01-20


In [12]:
import pandas as pd
import numpy as np

def normalizacion(df, columna_fecha):

    df_clean = df.copy()
    
    # 1. VALIDACION DE NULOS GENERALES
    nulos = df_clean.isna().sum()
    print("Conteo de nulos iniciales por columna:\n", nulos)

    # 2. FORMATEO IDENTIFICACION A MAYUSCULAS Y SIN ESPACIOS
    if 'identificacion' in df_clean.columns:
        df_clean['identificacion'] = df_clean['identificacion'].str.upper().str.strip()

    # 3. VALIDACION DE LAS FECHAS
    # convierte a datetime. errors='coerce' convierte fechas basura en Nulos
    df_clean[columna_fecha] = pd.to_datetime(df_clean[columna_fecha], errors='coerce')
    invalidas = df_clean[columna_fecha].isna()
    print(f"Cantidad de fechas inválidas o nulas ajustadas en '{columna_fecha}': {invalidas.sum()}")

    # 4. VALIDACION DE SALDO 
    if 'saldo' in df_clean.columns:
        # Convertir a numérico (por si hay letras o símbolos raros), los errores se vuelven NaN
        df_clean['saldo'] = pd.to_numeric(df_clean['saldo'], errors='coerce')
        
        # Rellenar los saldos NaN con 0
        df_clean['saldo'] = df_clean['saldo'].fillna(0)
        
        # Validar saldos negativos (El nivel N0 es Saldo >= 0
        negativos = (df_clean['saldo'] < 0).sum()
        if negativos > 0:
            print(f"Se encontraron {negativos} registros con saldo negativo. Ajustando a 0.")
            df_clean.loc[df_clean['saldo'] < 0, 'saldo'] = 0
            
        # Castear a entero para optimizar el almacenamiento en MySQL
        df_clean['saldo'] = df_clean['saldo'].astype(int)

    return df_clean





In [13]:
df_historia_limpio= normalizacion(saldo_historico,"corte_mes")
df_retiros_limpio= normalizacion(retiros,"fecha_retiro")
df_retiros_limpio

Conteo de nulos iniciales por columna:
 identificacion    0
corte_mes         0
saldo             0
dtype: int64
Cantidad de fechas inválidas o nulas ajustadas en 'corte_mes': 0
Conteo de nulos iniciales por columna:
 identificacion    0
fecha_retiro      0
dtype: int64
Cantidad de fechas inválidas o nulas ajustadas en 'fecha_retiro': 0


,identificacion,fecha_retiro
0,0TTW5R9RRCJ0A9E5F,2024-10-10
1,1LFTDYT2H0I9ZGED3S8,2023-11-11
2,6T645BU8TYM8VGXKO,2025-01-10
3,869YG73INQB5W1NQT,2024-12-05
4,THJ9OBJH3W6ANRCMS,2024-01-20


# CONEXION

In [14]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql

# 1. Configuración de credenciales
USER = 'root'
PASSWORD = '051099'
HOST = 'localhost'
PORT = '3306'
DATABASE = 'tuya_prueba_db' 

#1.1 conexion
print("Verificando existencia de la base de datos...")
try:
    # conexion al servidor
    conexion_server = pymysql.connect(host=HOST, user=USER, password=PASSWORD, port=int(PORT))
    cursor = conexion_server.cursor()
    
    # Ejecuta el comando de creación
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DATABASE};")
    conexion_server.commit()
    
    cursor.close()
    conexion_server.close()
    print(f"Base de datos '{DATABASE}' validada/creada con éxito.")
except Exception as e:
    print(f"Error al intentar crear la base de datos: {e}")

# 2. Carga a la Base de Datos con SQLAlchemy
cadena_conexion = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(cadena_conexion)


try:
    df_historia_limpio.to_sql(name='saldos_historicos', 
                              con=engine, 
                              if_exists='replace', 
                              index=False)
    
    df_retiros_limpio.to_sql(name='retiros', 
                              con=engine, 
                              if_exists='replace', 
                              index=False)
    
    print("Tablas cargadas exitosamente.")

except Exception as e:
    print(f"Ocurrió un error al cargar las tablas a MySQL: {e}")

Verificando existencia de la base de datos...
Base de datos 'tuya_prueba_db' validada/creada con éxito.
Tablas cargadas exitosamente.
